# UTUEL — Evaluation Notebook for Prompt Pipeline

Compile all pipeline output files into a single DataFrame, normalise ground-truth
and prediction values, then compute per-model accuracy metrics.

## 1 · Import Required Libraries

In [1]:
import subprocess
import sys
import json
from pathlib import Path

import pandas as pd

# Add project root to path so the package is importable from the notebook
PROJECT_ROOT = Path("../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from compile import (
    compile_all,
    compile_dataset,
    normalize_answer,
    is_correct,
)

DATASETS_DIR = PROJECT_ROOT / "datasets" /"results"
print(f"Project root : {PROJECT_ROOT}")
print(f"Datasets dir : {DATASETS_DIR}")

Project root : C:\Users\wtchuitc\Documents\GitHub\UTUEL
Datasets dir : C:\Users\wtchuitc\Documents\GitHub\UTUEL\datasets\results


## 2 · Compile Pipeline Outputs

Scan every `datasets/<name>/<model>/run<N>.jsonl` and merge into
`datasets/compiled_<name>/compiled.jsonl`.  
Already-compiled files are overwritten so results are always fresh.

In [2]:
# ── debug: show source run files, excluding compiled_* output folders ─────────
print(f"DATASETS_DIR exists : {DATASETS_DIR.exists()}")
print(f"DATASETS_DIR        : {DATASETS_DIR}\n")

# compile_all() discovers every dataset folder recursively and writes
# compiled_<name>/compiled.jsonl for each one, overwriting any previous run.
compiled_paths = compile_all(DATASETS_DIR)

print("\nCompiled files:")
if compiled_paths:
    for p in compiled_paths:
        print(f"  {p.relative_to(PROJECT_ROOT)}")
else:
    print("  (none — no *.jsonl files found yet)")

DATASETS_DIR exists : True
DATASETS_DIR        : C:\Users\wtchuitc\Documents\GitHub\UTUEL\datasets\results

[compile] test_lookup_WikiSQL → C:\Users\wtchuitc\Documents\GitHub\UTUEL\datasets\results\compiled_test_lookup_WikiSQL\compiled.jsonl  (70267 records)

Compiled files:
  datasets\results\compiled_test_lookup_WikiSQL\compiled.jsonl


## 3 · Load Compiled JSONL into a DataFrame

Load all compiled files and concatenate them into one DataFrame.

In [3]:
def load_compiled_jsonl(path: Path) -> pd.DataFrame:
    """Read a compiled JSONL file into a DataFrame."""
    records = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return pd.DataFrame(records)


# Discover all compiled.jsonl files under datasets/compiled_*/
# This works even if the compile cell above was not re-run this session.
compiled_files = sorted(DATASETS_DIR.glob("compiled_*/compiled.jsonl"))

if not compiled_files:
    raise RuntimeError(
        "No compiled.jsonl files found under datasets/compiled_*/\n"
        "Run the compile cell above first."
    )

print("Loading:")
frames = []
for p in compiled_files:
    frame = load_compiled_jsonl(p)
    print(f"  {p.relative_to(DATASETS_DIR)}  ({len(frame)} records)")
    frames.append(frame)

df = pd.concat(frames, ignore_index=True)

print(f"\nTotal records : {len(df)}")
print(f"Models        : {sorted(df['model'].unique())}")
print(f"Columns       : {list(df.columns)}")
df.head(3)

Loading:
  compiled_test_lookup_WikiSQL\compiled.jsonl  (70267 records)

Total records : 70267
Models        : ['TableGPT2-7B', 'deepseek-r1', 'gemma2', 'gemma4', 'gpt-oss', 'llama3', 'qwen3.6:27b', 'tablellm-7b-Q4']
Columns       : ['table_id', 'ground_truth', 'question', 'model', 'run', 'response', 'prediction', 'parse_ok', 'correct']


,table_id,ground_truth,question,model,run,response,prediction,parse_ok,correct
0,2-14642287-5,[1],How many bronze medals are there when there ar...,deepseek-r1,1,"{""answer"": ""1""}",1,True,True
1,2-15847138-2,"[Colonial Square, Masters]",Which Event has a 2007–08 of n/a?,deepseek-r1,1,"{""answer"": ""Colonial Square and Masters""}",Colonial Square and Masters,True,False
2,2-13771649-13,[32 - 08],What is the points difference associated with ...,deepseek-r1,1,"{""answer"": ""32 - 08""}",32 - 08,True,True


## 4 · Normalise Ground-Truth Values

`ground_truth` may be a `list[str]` (most rows) or a plain `str`.  
`normalize_answer()` from `compile.py` lower-cases, collapses whitespace, and
joins lists with `", "` so both sides of a comparison are in the same shape.

## 5 · Per-Model Accuracy Aggregation

Aggregate by `model` (and optionally `run`) to compute:
- **accuracy** — fraction of rows where `correct == True`
- **parse_rate** — fraction of rows where the JSON answer was successfully extracted
- **n** — total number of evaluated rows

In [4]:
summary = (
    df.groupby(["model", "run"])
    .agg(
        n          =("correct",  "count"),
        accuracy   =("correct",  "mean"),
        parse_rate =("parse_ok", "mean"),
    )
    .reset_index()
    .sort_values(["model", "run"])
)

summary["accuracy"]   = summary["accuracy"].map("{:.1%}".format)
summary["parse_rate"] = summary["parse_rate"].map("{:.1%}".format)

summary

,model,run,n,accuracy,parse_rate
0,TableGPT2-7B,1,11324,75.9%,99.8%
1,deepseek-r1,1,11324,68.8%,99.5%
2,gemma2,1,11324,69.5%,99.8%
3,gemma4,1,435,81.8%,100.0%
4,gpt-oss,1,11324,79.6%,100.0%
5,llama3,1,11324,56.8%,96.7%
6,qwen3.6:27b,1,1888,90.3%,99.9%
7,tablellm-7b-Q4,1,11324,22.9%,61.5%


### Overall accuracy collapsed across runs

In [5]:
overall = (
    df.groupby("model")
    .agg(
        n          =("correct",  "count"),
        accuracy   =("correct",  "mean"),
        parse_rate =("parse_ok", "mean"),
    )
    .reset_index()
    .sort_values("accuracy", ascending=False)
)

overall["accuracy"]   = overall["accuracy"].map("{:.1%}".format)
overall["parse_rate"] = overall["parse_rate"].map("{:.1%}".format)

overall

,model,n,accuracy,parse_rate
5,qwen3.6:27b,1170,91.3%,99.9%
3,gpt-oss,11324,79.6%,100.0%
0,TableGPT2-7B,11324,75.9%,99.8%
1,deepseek-r1,10390,69.9%,99.4%
2,gemma2,11324,69.5%,99.8%
4,llama3,11324,56.8%,96.7%
6,tablellm-7b-Q4,11324,22.9%,61.5%
